# ENN583 — Week 9 Practical: Representing 3D Scenes

Recovering depth is only the first part of building a useful 3D system. A robot, autonomous vehicle, or mixed-reality application must also decide **how to store the scene** and **how the rest of the system will query it**. Different representations preserve different information and make different operations easy.

In this practical, we begin with a real KITTI stereo pair and compare classical stereo depth with learned monocular depth from Depth Anything V2. We then reuse the recovered geometry in progressively more structured representations:

$$\text{images} \rightarrow \text{depth} \rightarrow \text{point cloud} \rightarrow \text{voxels} \rightarrow \text{learned implicit fields}$$

The goal is not just to produce attractive 3D visualisations. At each stage, ask what is stored, what has been discarded, how memory scales, and which questions the representation can answer. The final occupancy example uses a clean synthetic object because a single stereo image observes surfaces but does not provide trustworthy labels for the complete inside and outside of an object.

By the end of the practical, you should be able to:

- convert disparity into metric depth and back-project depth into 3D;
- explain the difference between explicit, discrete, and implicit representations;
- describe the resolution–memory trade-off of voxel grids;
- train and query continuous occupancy and signed-distance networks; and
- explain why relative monocular depth and calibrated stereo depth should not be interpreted on the same metric scale.

| Representation | What is stored? | Type |
|---|---|---|
| Point cloud | Observed surface samples | Explicit |
| Voxels | Occupied cells in a grid | Explicit and discrete |
| Occupancy field | Parameters of an inside/outside classifier | Implicit and learned |
| Signed-distance field | Parameters of a continuous distance function | Implicit and learned |


## 0. Setup

Viser provides the interactive 3D views in this practical. The next cell installs it only when it is missing.

In [ ]:
# Viser is not part of every ENN583 environment, so install it only if needed.
try:
    import viser
except ModuleNotFoundError:
    %pip install viser
    import viser

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from skimage.measure import marching_cubes
from torch import nn
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

# The course KITTI loader lives in the repository's support directory.
support_dir = (Path.cwd().resolve().parents[1] / "support")
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))
    
import kitti_utils as kitti

# Fix both random-number generators so repeated runs use the same samples.
SEED = 583
np.random.seed(SEED)
torch.manual_seed(SEED)
# PyTorch will use a GPU when one is available and otherwise fall back to CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Aim a new Viser view at the data instead of orbiting around the world origin.
def set_initial_camera(server, target, distance):
    target = np.asarray(target)
    server.initial_camera.look_at = tuple(target)
    server.initial_camera.position = tuple(target + np.array([0.0, -0.25 * distance, -distance]))
    server.initial_camera.up = (0.0, -1.0, 0.0)

## 1. KITTI stereo to dense depth

For rectified stereo images, corresponding pixels lie on the same image row. Their horizontal displacement is the disparity

$$d = u_L-u_R.$$

Depth follows from the focal length $f_x$ and camera baseline $B$:

$$Z=\frac{f_xB}{d}.$$

The stereo matcher is provided because our focus is the resulting 3D representation.

**Your turn:** complete the two short lines after the matcher: identify valid disparities, then convert them to metric depth. Before running the cell, predict where disparity will be least reliable. After viewing the result, identify examples of occlusion, weak texture, and excessive range.


In [ ]:
# Load one rectified colour stereo pair from the familiar KITTI sequence.
dataset = kitti.load_kitti_dataset("2011_09_26_drive_0035")
left_rgb, right_rgb = dataset.stereo(10)
# Writable copies avoid warnings when image libraries convert the arrays to tensors.
left_rgb = left_rgb.copy()
right_rgb = right_rgb.copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(left_rgb)
axes[0].set_title("Left image")
axes[1].imshow(right_rgb)
axes[1].set_title("Right image")
for axis in axes:
    axis.axis("off")
plt.show()

In [ ]:
# SGBM matches intensity patterns, so colour is not needed for disparity.
left_gray = cv2.cvtColor(left_rgb, cv2.COLOR_RGB2GRAY)
right_gray = cv2.cvtColor(right_rgb, cv2.COLOR_RGB2GRAY)

# numDisparities sets the search range and must be divisible by 16.
# P1 and P2 penalise small and large disparity changes respectively.
window_size = 5
matcher = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=128,
    blockSize=window_size,
    P1=8 * window_size**2,
    P2=32 * window_size**2,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=2,
    disp12MaxDiff=1,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY,
)
# OpenCV stores disparity with four fractional bits, hence the division by 16.
disparity = matcher.compute(left_gray, right_gray).astype(np.float32) / 16.0

# KITTI cameras 2 and 3 are the left and right rectified colour cameras.
left_calibration = dataset.camera_calibration(camera=2)
right_calibration = dataset.camera_calibration(camera=3)
K = left_calibration["K"]
P_left = left_calibration["P"]
P_right = right_calibration["P"]

# In a rectified projection matrix, P[0, 3] / P[0, 0] is the camera's x offset.
tx_left = P_left[0, 3] / P_left[0, 0]
tx_right = P_right[0, 3] / P_right[0, 0]
baseline = abs(tx_right - tx_left)

# Non-positive disparities do not produce a valid depth. 
# TODO: which disparities define points in front of the cameras?
# valid_disparity = ...  
# TODO: Implement this solution.
pass

depth = np.full_like(disparity, np.nan)
# TODO: apply Z = fB/d only at valid pixels.
# depth[valid_disparity] = ...
# TODO: Implement this solution.
pass

print(f"Focal length: {K[0, 0]:.2f} pixels")
print(f"Stereo baseline: {baseline:.3f} m")

In [ ]:
# Limit the display range so distant stereo noise does not dominate the colour scale.
MAX_DEPTH = 50.0

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
disparity_image = axes[0].imshow(
    np.where(valid_disparity, disparity, np.nan), cmap="magma"
)
axes[0].set_title("Disparity (pixels)")
fig.colorbar(disparity_image, ax=axes[0])

depth_image = axes[1].imshow(depth, cmap="viridis", vmin=0, vmax=MAX_DEPTH)
axes[1].set_title(f"Depth (clipped at {MAX_DEPTH:.0f} m)")
fig.colorbar(depth_image, ax=axes[1])
for axis in axes:
    axis.axis("off")
plt.show()

## 2. Stereo depth to a coloured point cloud

We now turn the metric stereo depth into an explicit 3D representation. For every valid pixel,

$$X=\frac{(u-c_x)Z}{f_x}, \qquad Y=\frac{(v-c_y)Z}{f_y}.$$

The RGB value at $(u,v)$ becomes the colour of the corresponding 3D point.

**Your turn:** complete the calculations for `x` and `y` in `back_project`. Explain why invalid depth must be removed before constructing the point cloud.

> **Viser controls:** when the inline 3D viewer appears, look in its **top-right corner**. The small buttons there open the scene controls. Drag in the main view to orbit, and scroll to zoom.


In [ ]:
# u and v contain the image coordinate of every pixel.
height, width = depth.shape
u, v = np.meshgrid(np.arange(width), np.arange(height))
# Read focal lengths and principal point from the intrinsic matrix.
fx, fy = K[0, 0], K[1, 1]
cx, cy = K[0, 2], K[1, 2]

# Back-project all valid pixels into the left camera coordinate frame.
def back_project(depth_image):
    valid = np.isfinite(depth_image) & (depth_image > 0) & (depth_image < MAX_DEPTH)
    z = depth_image[valid]
    # x = ...  # TODO: back-project horizontal pixel coordinates, use only the valid indices.
    # y = ...  # TODO: back-project vertical pixel coordinates, use only the valid indices.
    # TODO: Implement this solution.
    pass
    # Each row of xyz is one 3D point; use the matching image pixel as its colour.
    xyz = np.column_stack((x, y, z)).astype(np.float32)
    return xyz, left_rgb[valid]

# Keep all stereo points for later voxelisation.
stereo_points, stereo_colors = back_project(depth)

# Viser does not need every point to show the scene clearly in the browser.
def display_subset(points, colors, maximum=100_000):
    if len(points) <= maximum:
        return points, colors
    selected = np.random.default_rng(SEED).choice(len(points), maximum, replace=False)
    return points[selected], colors[selected]

display_stereo_points, display_stereo_colors = display_subset(stereo_points, stereo_colors)
print(f"Stereo points: {len(stereo_points):,}")

In [ ]:
# Visualise the metric stereo reconstruction on its own.
# The scene controls are behind the small buttons in the viewer's top-right corner.
stereo_server = viser.ViserServer()
stereo_target = np.median(display_stereo_points, axis=0)
set_initial_camera(stereo_server, stereo_target, distance=12.0)
stereo_server.scene.add_point_cloud(
    "/stereo/point-cloud",
    points=display_stereo_points,
    colors=display_stereo_colors,
    point_size=0.03,
)
stereo_server.scene.show()


A point cloud stores observed surface samples directly. It has no connectivity between samples and does not tell us whether unobserved space is free or occupied. Its density and missing regions also depend on the camera viewpoint and the stereo matcher.

Orbit around the reconstruction. Look for noisy depth at object boundaries, missing points in weakly textured regions, and unreliable distant geometry.


## 3. Relative monocular depth with Depth Anything V2

Stereo uses two calibrated images and gives metric depth in metres. Depth Anything V2 predicts scene structure from a single image and is available in two forms:

- **Relative-depth models** recover depth ordering and scene shape, but their output has no metric scale or offset. The model used below produces a disparity-like **relative inverse-depth** prediction: larger values indicate nearer regions.
- **Metric-depth models** are fine-tuned to predict metres. The official release provides indoor models trained on Hypersim with a 20 m range and outdoor models trained on Virtual KITTI 2 with an 80 m range. Small, Base, and Large variants are available for each domain.

The outdoor Small metric model would be the natural choice for KITTI. However, the official metric checkpoints use the Depth Anything project's own model implementation, whereas the relative Small checkpoint is available directly through Transformers. We use the Transformers-compatible relative model to keep the practical setup simple.

We deliberately do **not** align this prediction to stereo. Instead, we invert it and choose an arbitrary display scale. The resulting point cloud is useful for inspecting shape and depth ordering, but its coordinates are not metres and should not be compared numerically with the stereo cloud. The model weights are downloaded the first time the next cell runs.

#### Further information

- [Depth Anything V2 project page](https://depth-anything-v2.github.io/) — visual results and an overview of the project.
- [Depth Anything V2 paper](https://arxiv.org/abs/2406.09414) — model design, training strategy, and evaluation.
- [Official code repository](https://github.com/DepthAnything/Depth-Anything-V2) — source code, checkpoints, and examples.
- [Official metric-depth instructions](https://github.com/DepthAnything/Depth-Anything-V2/blob/main/metric_depth/README.md) — indoor and outdoor metric checkpoints and their usage.
- [Transformers documentation](https://huggingface.co/docs/transformers/model_doc/depth_anything_v2) — the API used in this notebook.
- [Depth Anything V2 Small model card](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf) — details of the exact relative-depth checkpoint loaded below.


In [ ]:
# The processor resizes and normalises the image exactly as the model expects.
DEPTH_MODEL = "depth-anything/Depth-Anything-V2-Small-hf"
depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL)
depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL).to(DEVICE)
# Evaluation mode disables training-only behaviour such as dropout.
depth_model.eval()

inputs = depth_processor(images=left_rgb, return_tensors="pt").to(DEVICE)
# Gradients are unnecessary for inference and would consume extra memory.
with torch.no_grad():
    prediction = depth_model(**inputs).predicted_depth

# Resize the lower-resolution prediction back to the KITTI image size.
relative_inverse_depth = torch.nn.functional.interpolate(
    prediction.unsqueeze(1),
    size=left_rgb.shape[:2],
    mode="bicubic",
    align_corners=False,
).squeeze().cpu().numpy()

plt.figure(figsize=(10, 4))
plt.imshow(relative_inverse_depth, cmap="magma")
plt.title("Depth Anything V2 relative inverse depth")
plt.colorbar(label="Relative inverse depth (arbitrary units)")
plt.axis("off")
plt.show()


In [ ]:
# Invert the positive prediction to obtain relative depth.
valid_relative = np.isfinite(relative_inverse_depth) & (relative_inverse_depth > 0)
relative_depth = np.full_like(relative_inverse_depth, np.nan)
relative_depth[valid_relative] = 1.0 / relative_inverse_depth[valid_relative]

# Relative depth has an arbitrary multiplicative scale. Choose one only so that
# the point cloud has a convenient size in the viewer; these units are not metres.
ARBITRARY_MEDIAN_DEPTH = 15.0
scale = ARBITRARY_MEDIAN_DEPTH / np.nanmedian(relative_depth)
arbitrary_depth = relative_depth * scale

# Remove extreme far values so they do not dominate the 3D visualisation.
arbitrary_depth[arbitrary_depth > 3 * ARBITRARY_MEDIAN_DEPTH] = np.nan

plt.figure(figsize=(10, 4))
plt.imshow(arbitrary_depth, cmap="viridis")
plt.title("Relative depth after inversion and arbitrary scaling")
plt.colorbar(label="Depth (arbitrary units)")
plt.axis("off")
plt.show()


### Relative-depth point cloud

We reuse the KITTI camera rays to lift the relative depth into 3D. This produces a helpful qualitative view of the predicted geometry, but it does not turn the monocular prediction into metric depth.

> **Viser controls:** open the controls from the small buttons in the **top-right corner** of this separate viewer. This scene has its own camera and scale; do not compare its coordinate values directly with the stereo scene.


In [ ]:
# Back-project the arbitrarily scaled monocular depth into a separate scene.
monocular_points, monocular_colors = back_project(arbitrary_depth)
display_monocular_points, display_monocular_colors = display_subset(
    monocular_points, monocular_colors
)

# Use a separate Viser server so no shared coordinate scale is implied.
relative_server = viser.ViserServer()
relative_target = np.median(display_monocular_points, axis=0)
set_initial_camera(relative_server, relative_target, distance=12.0)
relative_server.scene.add_point_cloud(
    "/depth-anything-v2/relative-point-cloud",
    points=display_monocular_points,
    colors=display_monocular_colors,
    point_size=0.03,
)
relative_server.scene.show()
print(f"Relative-depth points: {len(monocular_points):,}")


Inspect the road shape, object boundaries, thin structures, and missing regions. Compare these properties qualitatively with what you observed in the stereo viewer, while remembering that the two scenes have unrelated numerical scales.


## 4. Point cloud to voxels

A point cloud tells us where surfaces were observed, but many robotic tasks need to query whether regions of space are occupied. A voxel grid divides space into regularly sized cells.

> **Viser controls:** the scene controls are again opened from the small buttons in the **top-right corner** of the viewer.


In [ ]:
# Quantise continuous XYZ coordinates into integer voxel coordinates.
def voxelise(points, voxel_size):
    voxel_indices = np.floor(points / voxel_size).astype(np.int32)    
    # Multiple points may land in one cell; retain each occupied cell once.
    unique_indices = np.unique(voxel_indices, axis=0)    
    centres = (unique_indices + 0.5) * voxel_size    
    return centres.astype(np.float32)

voxel_sizes = [0.10, 0.25, 0.50]
voxel_sets = {}

# Compare detail and storage at three spatial resolutions.
for voxel_size in voxel_sizes:
    voxel_sets[voxel_size] = voxelise(stereo_points, voxel_size)
    # Estimate a dense grid covering the point-cloud bounding box.
    grid_shape = np.ceil(np.ptp(stereo_points, axis=0) / voxel_size).astype(int)
    dense_cells = int(np.prod(grid_shape))
    sparse_bytes = voxel_sets[voxel_size].shape[0] * 3 * 4
    print(
        f"Voxel size: {voxel_size:.2f} m | "
        f"occupied: {len(voxel_sets[voxel_size]):,} | "
        f"dense bool grid: {dense_cells / 1e6:.2f} MB | "
        f"sparse indices: {sparse_bytes / 1e6:.2f} MB"
    )

In [ ]:
# Build all voxel cubes as one mesh rather than thousands of Viser objects.
# If the controls are hidden, use the small buttons in the viewer's top-right corner.
def cube_mesh(centres, size):
    corners = np.array([
        [-1, -1, -1], [1, -1, -1], [1, 1, -1], [-1, 1, -1],
        [-1, -1, 1], [1, -1, 1], [1, 1, 1], [-1, 1, 1],
    ], dtype=np.float32) * (size / 2)
    cube_faces = np.array([
        [0, 1, 2], [0, 2, 3], [4, 6, 5], [4, 7, 6],
        [0, 4, 5], [0, 5, 1], [1, 5, 6], [1, 6, 2],
        [2, 6, 7], [2, 7, 3], [3, 7, 4], [3, 4, 0],
    ], dtype=np.uint32)

    # Broadcasting places the same eight local corners around every centre.
    vertices = (centres[:, None, :] + corners[None, :, :]).reshape(-1, 3)
    # Offset the face indices so each cube refers to its own eight vertices.
    offsets = (8 * np.arange(len(centres), dtype=np.uint32))[:, None, None]
    faces = (cube_faces[None, :, :] + offsets).reshape(-1, 3)
    return vertices, faces

# Change this value to one of the resolutions computed above.
DISPLAY_VOXEL_SIZE = 0.25
displayed_voxels = voxel_sets[DISPLAY_VOXEL_SIZE]
voxel_vertices, voxel_faces = cube_mesh(displayed_voxels, DISPLAY_VOXEL_SIZE)

voxel_server = viser.ViserServer()
set_initial_camera(voxel_server, np.median(displayed_voxels, axis=0), distance=12.0)
voxel_server.scene.add_mesh_simple(
    "/voxels/wireframe",
    vertices=voxel_vertices,
    faces=voxel_faces,
    color=(40, 120, 220),
    wireframe=True,
)
voxel_server.scene.show()

Change `DISPLAY_VOXEL_SIZE` and rerun the visualisation cell to inspect another resolution.

1. What is lost when the voxel size increases?
2. Why does halving the voxel width potentially require eight times as many cells in a dense 3D grid?
3. Which representation makes a direct occupancy lookup easier: a point cloud or a voxel grid?

## 5. Learned implicit fields: occupancy and signed distance

A voxel grid stores values at predefined locations. Can we instead learn a continuous function that can be queried anywhere in 3D? We will compare two targets using the same MLP architecture:

$$f_{\text{occ}}(x,y,z) \rightarrow \text{occupancy logit}$$

$$f_{\text{sdf}}(x,y,z) \rightarrow \text{signed distance}$$

The occupancy field learns only whether a coordinate is inside or outside. The signed-distance field additionally learns how far the coordinate is from the surface, with negative values inside and positive values outside. Both surfaces are obtained from a level set: occupancy logit $=0$ (probability $=0.5$), or SDF $=0$.

We use an **analytical signed-distance function** for a clean synthetic sphere–box union. For the occupancy network, its sign is converted into binary labels; the network never sees the distance values. For the SDF network, the distance itself is the regression target. This avoids implying that a single stereo view provides complete inside/outside or signed-distance supervision.

Our second model demonstrates the central SDF-regression idea used by DeepSDF. Full DeepSDF also learns a latent code for each shape across a collection of shapes; this small example learns only one shape.

**Your turn:** supply the final layer shared by both network architectures. It must map 64 hidden features to one scalar. Why does the occupancy model return an unbounded logit rather than applying a sigmoid itself? How is the SDF model's scalar interpreted differently?


In [ ]:
# This analytical SDF defines the synthetic training object.
# Negative distance means inside; positive distance means outside.
def object_signed_distance(xyz):
    sphere_distance = np.linalg.norm(xyz - np.array([-0.25, 0.0, 0.0]), axis=1) - 0.55

    # Signed distance to an axis-aligned box centred at (0.35, 0, 0).
    box_offset = np.abs(xyz - np.array([0.35, 0.0, 0.0])) - np.array([0.45, 0.32, 0.32])
    box_distance = (
        np.linalg.norm(np.maximum(box_offset, 0), axis=1)
        + np.minimum(np.max(box_offset, axis=1), 0)
    )
    # The minimum distance represents the union of the sphere and box.
    return np.minimum(sphere_distance, box_distance)

rng = np.random.default_rng(SEED)
# Generate a large candidate pool throughout the object's bounding cube.
candidates = rng.uniform(-1.0, 1.0, size=(200_000, 3)).astype(np.float32)
candidate_distance = object_signed_distance(candidates)

# Balance inside/outside samples and include many points close to the surface.
samples_per_class = 10_000
sample_parts = []
for occupied in [False, True]:
    class_indices = np.flatnonzero((candidate_distance <= 0) == occupied)
    near_indices = class_indices[np.abs(candidate_distance[class_indices]) < 0.08]
    near = rng.choice(near_indices, samples_per_class // 2, replace=False)
    broad = rng.choice(class_indices, samples_per_class // 2, replace=False)
    sample_parts.append(candidates[np.concatenate((near, broad))])

sample_points = np.concatenate(sample_parts)
sample_distances = object_signed_distance(sample_points).astype(np.float32)
sample_labels = (sample_distances <= 0).astype(np.float32)

# DeepSDF-style training commonly truncates large distances to focus on the surface.
SDF_TRUNCATION = 0.20
sample_distances = np.clip(sample_distances, -SDF_TRUNCATION, SDF_TRUNCATION)

# Shuffle once, then hold out 20% of the samples for validation.
order = rng.permutation(len(sample_points))
split = int(0.8 * len(order))
train_indices, validation_indices = order[:split], order[split:]

# Both models receive the same coordinates but different supervision.
train_xyz = torch.from_numpy(sample_points[train_indices]).to(DEVICE)
validation_xyz = torch.from_numpy(sample_points[validation_indices]).to(DEVICE)
train_labels = torch.from_numpy(sample_labels[train_indices, None]).to(DEVICE)
validation_labels = torch.from_numpy(sample_labels[validation_indices, None]).to(DEVICE)
train_sdf = torch.from_numpy(sample_distances[train_indices, None]).to(DEVICE)
validation_sdf = torch.from_numpy(sample_distances[validation_indices, None]).to(DEVICE)

print(f"Training samples: {len(train_indices):,}")
print(f"Validation samples: {len(validation_indices):,}")
print(f"Occupied samples: {sample_labels.mean():.1%}")


In [ ]:
# Both fields use the same architecture but learn different target functions.

class FieldMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, xyz):
        return self.network(xyz)

occupancy_model = FieldMLP().to(DEVICE)
sdf_model = FieldMLP().to(DEVICE)

# Occupancy is binary classification; SDF prediction is scalar regression.
occupancy_loss_function = nn.BCEWithLogitsLoss()
sdf_loss_function = nn.L1Loss()
occupancy_optimiser = torch.optim.Adam(occupancy_model.parameters(), lr=1e-3)
sdf_optimiser = torch.optim.Adam(sdf_model.parameters(), lr=1e-3)


In [ ]:
epochs = 1500
occupancy_losses = []
sdf_losses = []

# Train the two independent fields on the same coordinates.
occupancy_model.train()
sdf_model.train()
for epoch in range(epochs):
    # The occupancy model learns binary inside/outside labels.
    occupancy_optimiser.zero_grad()
    occupancy_logits = occupancy_model(train_xyz)
    occupancy_loss = occupancy_loss_function(occupancy_logits, train_labels)
    occupancy_loss.backward()
    occupancy_optimiser.step()
    occupancy_losses.append(occupancy_loss.item())

    # The SDF model regresses truncated signed-distance values.
    sdf_optimiser.zero_grad()
    predicted_sdf = sdf_model(train_xyz)
    sdf_loss = sdf_loss_function(predicted_sdf, train_sdf)
    sdf_loss.backward()
    sdf_optimiser.step()
    sdf_losses.append(sdf_loss.item())

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
axes[0].plot(occupancy_losses)
axes[0].set_title("Occupancy classification")
axes[0].set_ylabel("Binary cross-entropy")
axes[1].plot(sdf_losses)
axes[1].set_title("Signed-distance regression")
axes[1].set_ylabel("Mean absolute error")
for axis_plot in axes:
    axis_plot.set_xlabel("Epoch")
    axis_plot.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Evaluate each model using a metric appropriate for its target.
occupancy_model.eval()
sdf_model.eval()
with torch.no_grad():
    occupancy_predictions = torch.sigmoid(occupancy_model(validation_xyz)) > 0.5
    occupancy_targets = validation_labels.bool()
    accuracy = (occupancy_predictions == occupancy_targets).float().mean()
    intersection = (occupancy_predictions & occupancy_targets).sum()
    union = (occupancy_predictions | occupancy_targets).sum()
    iou = intersection / union
    sdf_mae = torch.mean(torch.abs(sdf_model(validation_xyz) - validation_sdf))

print(f"Occupancy validation accuracy: {accuracy.item():.1%}")
print(f"Occupancy validation IoU: {iou.item():.3f}")
print(f"SDF validation MAE: {sdf_mae.item():.4f}")


## 6. Query and compare the learned fields

Both trained networks can be evaluated at any 3D coordinate. We query them on a regular grid only to visualise their learned zero-level surfaces; the networks themselves are not tied to this grid resolution.

The occupancy model is converted to probability for interpretation, then contoured at $p=0.5$. The SDF model is contoured directly at distance zero. Similar surfaces do not mean the learned functions are identical: away from the boundary, one represents class confidence while the other represents signed distance.

> **Viser controls:** use the small buttons in the **top-right corner** of the viewer to open the scene controls and toggle the analytical, occupancy, and SDF surfaces.


In [ ]:
# This grid is only for visualisation; both MLPs remain continuous.
grid_resolution = 64
axis = np.linspace(-1.0, 1.0, grid_resolution, dtype=np.float32)
grid_xyz = np.stack(np.meshgrid(axis, axis, axis, indexing="ij"), axis=-1)
flat_grid = grid_xyz.reshape(-1, 3)

probability_parts = []
sdf_parts = []
occupancy_model.eval()
sdf_model.eval()
with torch.no_grad():
    # Query in chunks to avoid placing the complete dense grid on the GPU.
    for start in range(0, len(flat_grid), 65_536):
        query = torch.from_numpy(flat_grid[start:start + 65_536]).to(DEVICE)
        probability_parts.append(torch.sigmoid(occupancy_model(query)).cpu().numpy())
        sdf_parts.append(sdf_model(query).cpu().numpy())

shape = (grid_resolution, grid_resolution, grid_resolution)
occupancy_probabilities = np.concatenate(probability_parts).reshape(shape)
predicted_sdf_grid = np.concatenate(sdf_parts).reshape(shape)
spacing = 2.0 / (grid_resolution - 1)

# Occupancy extracts p=0.5; SDF extracts signed distance=0.
occupancy_vertices, occupancy_faces, _, _ = marching_cubes(
    occupancy_probabilities,
    level=0.5,
    spacing=(spacing,) * 3,
    gradient_direction="ascent",
)
sdf_vertices, sdf_faces, _, _ = marching_cubes(
    predicted_sdf_grid,
    level=0.0,
    spacing=(spacing,) * 3,
    gradient_direction="descent",
)

# Extract the analytical SDF zero level as the reference surface.
ground_truth_sdf = object_signed_distance(flat_grid).reshape(shape)
truth_vertices, truth_faces, _, _ = marching_cubes(
    ground_truth_sdf,
    level=0.0,
    spacing=(spacing,) * 3,
    gradient_direction="descent",
)

# Marching Cubes starts coordinates at zero; shift back to [-1, 1].
for vertices in (occupancy_vertices, sdf_vertices, truth_vertices):
    vertices -= 1.0

print(f"Occupancy surface: {len(occupancy_vertices):,} vertices")
print(f"SDF surface: {len(sdf_vertices):,} vertices")


In [ ]:
# Place the analytical and two learned surfaces side by side.
# The visibility controls are behind the small buttons in the viewer's top-right corner.
field_server = viser.ViserServer()
set_initial_camera(field_server, target=(0.0, 0.0, 0.0), distance=6.0)
field_server.scene.add_mesh_simple(
    "/analytical-ground-truth",
    vertices=truth_vertices,
    faces=truth_faces,
    color=(80, 80, 80),
    position=(-2.4, 0.0, 0.0),
)
field_server.scene.add_mesh_simple(
    "/learned-occupancy/zero-level",
    vertices=occupancy_vertices,
    faces=occupancy_faces,
    color=(70, 150, 230),
)
field_server.scene.add_mesh_simple(
    "/learned-sdf/zero-level",
    vertices=sdf_vertices,
    faces=sdf_faces,
    color=(230, 140, 60),
    position=(2.4, 0.0, 0.0),
)

# Add world-space anchors with screen-sized text above each surface.
# Negative Y appears upward because this viewer uses (0, -1, 0) as camera up.
surface_labels = [
    ("ground-truth", "Ground truth", -2.4),
    ("occupancy", "Occupancy network", 0.0),
    ("sdf", "SDF network", 2.4),
]
for label_name, label_text, x_position in surface_labels:
    field_server.scene.add_label(
        f"/labels/{label_name}",
        label_text,
        position=(x_position, -0.9, 0.0),
        anchor="bottom-center",
        font_size_mode="screen",
        font_screen_scale=1.2,
        depth_test=False,
    )
field_server.scene.show()


## 7. Compare the representations

For each question, identify the best representation and explain why.

1. Which representation preserves the exact surface samples recovered from the camera?
2. Which representation makes collision or occupancy lookup simplest?
3. Which representations can be queried continuously at arbitrary coordinates?
4. Which representation has memory that grows directly with grid resolution?
5. What information from the KITTI image was discarded by the synthetic implicit-field experiment?
6. What does the occupancy network learn from the analytical SDF, and what information is discarded when distances become binary labels?
7. How do the occupancy and SDF losses differ, and why is sigmoid used only when interpreting occupancy logits?
8. Full DeepSDF uses a latent code per shape. What would that add beyond this single-shape SDF network?
9. Why can the two KITTI point clouds be compared qualitatively but not by their numerical coordinates?

### Take-away

A point cloud explicitly stores observed samples. A voxel grid explicitly stores values at discrete spatial locations and makes occupancy lookup straightforward, but its cost grows rapidly with resolution. An occupancy field learns a continuous inside/outside classifier, whereas an SDF field learns a continuous signed distance whose zero level defines the surface. Both are implicit representations, but their supervision, losses, outputs, and useful queries differ. Changing representation does not automatically repair errors or recover information absent from the observations.
